In [ ]:
!pip install transformers torch pandas scipy --quiet

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import torch

In [ ]:
from google.colab.files import upload
upload()

Saving Nallur Kovil.csv to Nallur Kovil.csv


{'Nallur Kovil.csv': b'\xef\xbb\xbflocation_name,location_id,review_text,rating,published_date,language\r\nNallur Kovil,551304,"Not your average temple & the main temple in Jaffna to visit. Simple colour and tones and minimal god statues. They still only charge 1 rupee to do a service. No photography, no going on your phone - You have to love the authentic / no change mentality in such a changing tech heavy world. You must cover your legs and arms if female. Must cover your legs & wear no top if male. Great experience! When a pooja is on you are literally running around the whole temple following the priest as they do a service with each god.",,Dec 2025,en\r\nNallur Kovil,551304,"Personally, I didn\'t like the Nallur Kovil so much, it\'s kept rather simple inside. From the location on the road also rather not as a highlight as the Koneshwaram Kovil in Trinco, which is perched high on a rock. I found the experience rather disappointing",,Nov 2024,en\r\nNallur Kovil,551304,This review is

In [ ]:
df = pd.read_csv("/content/Nallur Kovil.csv")

In [ ]:
df = df.dropna(subset=["review_text"]).reset_index(drop=True)

In [ ]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
labels = ["Negative", "Neutral", "Positive"]

In [ ]:
def get_sentiment(text):

    encoded_input = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)


    with torch.no_grad():
        output = model(**encoded_input)


    scores = output.logits[0].numpy()
    scores = softmax(scores)


    sentiment_label = labels[scores.argmax()]
    sentiment_score = scores.max()

    return sentiment_label, sentiment_score, scores[0], scores[1], scores[2]

In [ ]:
results = df["review_text"].apply(get_sentiment)

In [ ]:
df["sentiment_label"] = results.apply(lambda x: x[0])
df["sentiment_score"] = results.apply(lambda x: x[1])
df["negative_score"] = results.apply(lambda x: x[2])
df["neutral_score"] = results.apply(lambda x: x[3])
df["positive_score"] = results.apply(lambda x: x[4])

In [ ]:
print(df[["review_text", "sentiment_label", "sentiment_score"]].head())

                                         review_text sentiment_label  \
0  Not your average temple & the main temple in J...        Positive   
1  Personally, I didn't like the Nallur Kovil so ...        Negative   
2  This review is the subjective opinion of a Tri...         Neutral   
3  Great temple, beautiful architecture, huge pal...        Positive   
4  This review is the subjective opinion of a Tri...         Neutral   

   sentiment_score  
0         0.740091  
1         0.950855  
2         0.777266  
3         0.817138  
4         0.777266  


In [ ]:
df.to_csv("Nallur_Kovil_with_sentiment.csv", index=False)
print("Saved as Nallur_Kovil_with_sentiment.csv")

Saved as Nallur_Kovil_with_sentiment.csv


In [ ]:
df = pd.read_csv("Nallur_Kovil_with_sentiment.csv")

In [ ]:
df["published_date"] = pd.to_datetime(df["published_date"], format="%b %Y")

In [ ]:
df["year_month"] = df["published_date"].dt.to_period("M")

In [ ]:
monthly_counts = (
    df.groupby(["year_month", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)


In [ ]:

for col in ["Positive", "Negative", "Neutral"]:
    if col not in monthly_counts.columns:
        monthly_counts[col] = 0

In [ ]:
monthly_avg_score = df.groupby("year_month")["sentiment_score"].mean()
monthly_counts["avg_sentiment_score"] = monthly_avg_score

In [ ]:
monthly_counts["total_reviews"] = monthly_counts[["Positive", "Negative", "Neutral"]].sum(axis=1)

monthly_counts["sentiment_index"] = (
    (monthly_counts["Positive"] - monthly_counts["Negative"]) / monthly_counts["total_reviews"]
)

In [ ]:
def label_from_index(index_value):
    if index_value > 0.05:
        return "Positive"
    elif index_value < -0.05:
        return "Negative"
    else:
        return "Neutral"

monthly_counts["final_monthly_label"] = monthly_counts["sentiment_index"].apply(label_from_index)

In [ ]:
monthly_summary = monthly_counts.reset_index()
monthly_summary = monthly_summary.sort_values("year_month")

print(monthly_summary[[
    "year_month", "Positive", "Negative", "Neutral",
    "total_reviews", "avg_sentiment_score", "sentiment_index", "final_monthly_label"
]])

sentiment_label year_month  Positive  Negative  Neutral  total_reviews  \
0                  2011-09         1         0        0              1   
1                  2011-10         0         0        1              1   
2                  2011-11         1         0        0              1   
3                  2011-12         0         0        1              1   
4                  2012-04         0         0        1              1   
..                     ...       ...       ...      ...            ...   
109                2024-11         0         1        0              1   
110                2025-01         0         0        1              1   
111                2025-03         1         0        0              1   
112                2025-06         0         0        1              1   
113                2025-12         1         0        0              1   

sentiment_label  avg_sentiment_score  sentiment_index final_monthly_label  
0                           0.95242

In [ ]:
monthly_summary.to_csv("Nallur_Kovil_monthly_sentiment_index.csv", index=False)
print("Saved as Nallur_Kovil_monthly_sentiment_index.csv")

Saved as Nallur_Kovil_monthly_sentiment_index.csv
